# Notebook 3: Logistic Regression (Probabilities)

In the previous notebook, it was observed that the **Perceptron** has limitations:
1.  It is a rigid binary classifier (yes/no).
2.  It does not handle non-linearly separable data well.
3.  It does not indicate "how confident" it is in its decision.

In this lab, **Logistic Regression** will be implemented. Despite its name, it is a classification algorithm that utilizes the **Sigmoid** function to estimate the probability that a drawing belongs to a class.

### Objective
To substitute the abrupt decision of the Perceptron with a **probability estimation** ($P(y=1|x)$) to improve the robustness of the *Doodles* recognition system.

Run the following commands to import the necessary libraries and set up the working environment.

In [45]:
!pip install seaborn scikit-learn pandas pillow gradio matplotlib

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\hecto\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


<p style="page-break-after:always;"></p>

# Logistic Regression

In [46]:
import numpy as np

**OneHotEncoding:** $\;$ function that converts class labels to *one-hot* representation

$$\operatorname{one-hot}(y)%
=\boldsymbol{y}%
=\begin{pmatrix}y_1\\\vdots\\y_C\end{pmatrix}%
=\begin{pmatrix}\mathbb{I}(y=1)\\\vdots\\\mathbb{I}(y=C)\end{pmatrix}\in\{0,1\}^C%
\quad\text{with}\quad%
\sum_c y_c=1$$

$\;$ where $y$ is a categorical variable taking a value from $C$ possible categories, $\,\{1,\dotsc,C\}$

In [47]:
def OneHotEncoding(y):
  # Unique class labels
  c = np.unique(y); C = c.size;

  # Map class labels from 0 to C-1
  # Create a mapping from values of c to their corresponding indices
  mapping = {value: idx for idx, value in enumerate(c)}
  # Convert y using this mapping
  y = np.vectorize(mapping.get)(y)

  # Generate "one-hot" representation using the np.eye method
  return np.eye(C)[y]

**ComputeGradient:** $\;$ weight update for logistic regression

Gradient descent applied to logistic regression:

Neg-log-likelihood (NLL): $\;$ the neg-log-likelihood is a convex objective function
$$\operatorname{NLL}(\mathbf{W})%
=\frac{1}{N}\sum_{n=1}^N-\log p(\boldsymbol{y}_n\mid\boldsymbol{\mu}_n)
\qquad\text{with}\quad%
\boldsymbol{\mu}_n=\mathcal{S}(\boldsymbol{a}_n)%
\quad\text{and}\quad%
\boldsymbol{a}_n=\mathbf{W}^t\boldsymbol{x}_n$$

NLL Gradient:
$$\begin{pmatrix}%
\frac{\partial\operatorname{NLL}}{\partial W_{11}}&\cdots&\frac{\partial\operatorname{NLL}}{\partial W_{1C}}\\%
\vdots&\ddots&\vdots\\%
\frac{\partial\operatorname{NLL}}{\partial W_{D1}}&\cdots&\frac{\partial\operatorname{NLL}}{\partial W_{DC}}\\%
\end{pmatrix}
=\frac{\partial\operatorname{NLL}}{\partial\mathbf{W}^t}%
=\frac{1}{N}\sum_{n=1}^N\frac{\partial(-\log p(\boldsymbol{y}_n\mid\boldsymbol{\mu}_n))}{\partial \mathbf{W}^t}%
=\frac{1}{N}\sum_{n=1}^N\boldsymbol{x}_n(\boldsymbol{\mu}_n-\boldsymbol{y}_n)^t$$

In [48]:
def ComputeGradient(X, Y, W):
  N, D = X.shape
  X = np.concatenate((np.ones((N,1)), X),axis=1)
  N, C = Y.shape
  Z = np.zeros((N, C)).astype(float)

  # Calculate logits
  a = X @ W
  # Normalize logits for robust computation to avoid overflows
  a -= a.max(axis=1, keepdims=True)
  # Calculate softmax (probabilistic class label)
  mu = np.exp(a)
  mu /= np.sum(mu, axis=1, keepdims=True)
  # Subtract the current label from the probabilistic class label
  Z = mu - Y
  # The gradient is the summation, for all samples, of the outer dot product
  return (X.T @ Z)/N

<p style="page-break-after:always;"></p>

**LogisticRegressionClassification:** $\;$ classification of samples according to a given weight matrix

$$c(\boldsymbol{x})%
=\operatorname*{argmax}\limits_c\;\boldsymbol{\mu}_c%
\quad\text{with}\quad%
\boldsymbol{\mu}_c=\mathcal{S}(\boldsymbol{a})_c,\quad%
\boldsymbol{a}=f(\boldsymbol{x};\mathbf{W})=\mathbf{W}^t\boldsymbol{x},\quad%
\mathbf{W}\in\mathbb{R}^{D\times C}\quad\text{and}\quad\boldsymbol{x}\in\mathbb{R}^D$$

In [49]:
def LogisticRegressionClassification(X, W):
  N, D = X.shape
  X = np.concatenate((np.ones((N,1)), X),axis=1)

  # Calculate logits
  a = X @ W
  # Normalize logits for robust computation to avoid overflows
  a -= a.max(axis=1, keepdims=True)
  # Calculate softmax (probabilistic class label)
  mu = np.exp(a)
  mu /= np.sum(mu, axis=1, keepdims=True)
  # The class is the argument of the maximum of the probabilistic class label
  return np.argmax(mu,axis=1)

**LogisticRegressionTraining:** $\;$ batch training implementation based on stochastic gradient descent

Gradient descent applied to logistic regression: $\displaystyle\quad\mathbf{W}_0=\mathbf{0};\quad\mathbf{W}_{i+1}=\mathbf{W}_i -\eta_i\frac{\partial\operatorname{NLL}}{\partial\mathbf{W}^t}\biggr\vert_{\mathbf{W}_i}\quad i=0,1,\ldots$

In [50]:
def LogisticRegressionTraining(X, y, W=None, bs=1, maxEpochs=10, eta=1e-2, tol=1e-3):
  rng = np.random.default_rng(seed=23)
  N, D = X.shape;
  Y = OneHotEncoding(y)
  N, C = Y.shape
  if W is None:  # W_0 = 0
    W = np.zeros((1+D, C))
  grad = np.inf; epoch = 0
  while np.max(np.abs(grad)) > tol and epoch < maxEpochs:  # W_i+1 = W_i - eta_i * grad_W_i
    perm = rng.permutation(N); Xperm = X[perm]; Yperm = Y[perm]
    j = 0
    for j in range(N//bs):
      X_batch = Xperm[j*bs:(j+1)*bs]
      Y_batch = Yperm[j*bs:(j+1)*bs]
      grad = ComputeGradient(X_batch, Y_batch, W)
      W = W - eta*grad
    # If samples remain after batch partitioning
    if((j+1)*bs < N):
      X_batch = Xperm[(j+1)*bs:]
      Y_batch = Yperm[(j+1)*bs:]
      grad = ComputeGradient(X_batch, Y_batch, W)
      W = W - eta*grad
    epoch += 1
  return W

# Exercise: Logistic Regression applied to MyDoodles

The dataset is loaded and partitioned. Subsequently, the maximum number of training epochs is adjusted to observe the evolution of the error.

**We read and partition the dataset:**

In [51]:
import numpy as np
from sklearn.model_selection import train_test_split
import os

# Paths to the files (Ensure the path is correct)
path_X = '../data/doodles_X.npy'
path_y = '../data/doodles_y.npy'

# Data reading "teacher" style
with open(path_X, 'rb') as fd:
    X = np.load(fd)

with open(path_y, 'rb') as fd:
    y = np.load(fd).astype(int) # Ensure they are integers

# Dataset partition (Random State 23 as in the image)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=True,
    random_state=23
)

# Define N and M as requested by the exercise
N = len(X_train)
M = len(X_test)

print(f"Data loaded: N={N} (Train), M={M} (Test)")

Data loaded: N=80 (Train), M=20 (Test)


**Exercise: We adjust the maximum number of training epochs:**

In [52]:
print(f"{'bs':>4} {'maxEps':>8} {'eta':>8} {'trainErr':>10} {'testErr':>10}")
print("-" * 45)

# Initial exercise configuration
bs = 10
eta = 1e-2

# Loop to test different epochs
for maxEpochs in (5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000):

    # 1. Train
    W = LogisticRegressionTraining(X_train, y_train, bs=bs, maxEpochs=maxEpochs, eta=eta)

    # 2. Calculate training error
    haty_train = LogisticRegressionClassification(X_train, W)
    acc_train = np.sum(haty_train == y_train) / N

    # 3. Calculate test error
    haty_test = LogisticRegressionClassification(X_test, W)
    acc_test = np.sum(haty_test == y_test) / M

    # 4. Print results (Error = 1.0 - Accuracy)
    print(f"{bs:4d} {maxEpochs:8d} {eta:.1e} {1.0-acc_train:9.1%} {1.0-acc_test:9.1%}")

  bs   maxEps      eta   trainErr    testErr
---------------------------------------------
  10        5 1.0e-02     25.0%     70.0%
  10       10 1.0e-02     15.0%     65.0%
  10       20 1.0e-02     13.7%     65.0%
  10       50 1.0e-02      2.5%     55.0%
  10      100 1.0e-02      0.0%     35.0%
  10      200 1.0e-02      0.0%     30.0%
  10      500 1.0e-02      0.0%     20.0%
  10     1000 1.0e-02      0.0%     20.0%
  10     2000 1.0e-02      0.0%     20.0%
  10     5000 1.0e-02      0.0%     20.0%


**Exercise: We adjust the learning rate or *learning rate* ($\eta$/eta):**

In [53]:
print(f"{'bs':>4} {'maxEps':>8} {'eta':>8} {'trainErr':>10} {'testErr':>10}")
print("-" * 45)

# Fixed configuration for this experiment
bs = 10
maxEpochs = 500

# Testing powers of 10 for eta
for eta in (1e-4, 1e-3, 1e-2, 1e-1, 1, 1e1, 1e2):

    # 1. Train
    W = LogisticRegressionTraining(X_train, y_train, bs=bs, maxEpochs=maxEpochs, eta=eta)

    # 2. Validate (Train)
    haty_train = LogisticRegressionClassification(X_train, W)
    acc_train = np.sum(haty_train == y_train) / N

    # 3. Validate (Test)
    haty_test = LogisticRegressionClassification(X_test, W)
    acc_test = np.sum(haty_test == y_test) / M

    # 4. Print
    print(f"{bs:4d} {maxEpochs:8d} {eta:.1e} {1.0-acc_train:9.1%} {1.0-acc_test:9.1%}")

  bs   maxEps      eta   trainErr    testErr
---------------------------------------------
  10      500 1.0e-04     25.0%     70.0%
  10      500 1.0e-03      2.5%     55.0%
  10      500 1.0e-02      0.0%     20.0%
  10      500 1.0e-01      0.0%     20.0%
  10      500 1.0e+00      0.0%     25.0%
  10      500 1.0e+01      0.0%     30.0%
  10      500 1.0e+02      2.5%     70.0%


**Ejerc.: Ajustamos tamaño de lote (*batch size*):** parámetro *bs* define cada cuántas muestras se actualiza la matriz de pesos

In [54]:
print(f"{'bs':>4} {'maxEps':>8} {'eta':>8} {'trainErr':>10} {'testErr':>10}")
print("-" * 45)

# The best values found previously are used
maxEpochs = 500
eta = 1.0e-01  # CORRECTION: The image shows +02, but -02 is used as it performed better

# Different batch sizes are tested
for bs in (1, 2, 5, 10, 20, 50, 100):

    # 1. Train
    W = LogisticRegressionTraining(X_train, y_train, bs=bs, maxEpochs=maxEpochs, eta=eta)

    # 2. Validate (Train)
    haty_train = LogisticRegressionClassification(X_train, W)
    acc_train = np.sum(haty_train == y_train) / N

    # 3. Validate (Test)
    haty_test = LogisticRegressionClassification(X_test, W)
    acc_test = np.sum(haty_test == y_test) / M

    # 4. Print
    print(f"{bs:4d} {maxEpochs:8d} {eta:.1e} {1.0-acc_train:9.1%} {1.0-acc_test:9.1%}")

  bs   maxEps      eta   trainErr    testErr
---------------------------------------------
   1      500 1.0e-01      0.0%     20.0%
   2      500 1.0e-01      0.0%     20.0%
   5      500 1.0e-01      0.0%     20.0%
  10      500 1.0e-01      0.0%     20.0%
  20      500 1.0e-01      0.0%     20.0%
  50      500 1.0e-01      0.0%     20.0%
 100      500 1.0e-01     88.8%     95.0%


**Exercise: Final classifier:** $\;$ We train the final classifier with all available data and with the best observed parameters. Then we save it, and load it to evaluate

In [55]:
# Exercise: Final Classifier
print("Training definitive model with ALL data...")

# 1. WINNING Configuration
# (Substitute these values if better numbers were observed in the previous table)
bs = 10
maxEpochs = 500
eta = 1.0e-01  # 0.1

# 2. Training is performed using X and y (the complete dataset, without splitting)
W_final = LogisticRegressionTraining(X, y, bs=bs, maxEpochs=maxEpochs, eta=eta)

# 3. The weights are saved
# Saved in the models folder to maintain order
if not os.path.exists('../models'):
    os.makedirs('../models')

model_path = '../models/logistic_weights.npy'
np.save(model_path, W_final)

print(f"Model successfully saved at: {model_path}")

Training definitive model with ALL data...
Model successfully saved at: ../models/logistic_weights.npy


In [56]:
print("Verifying model loading...")

model_path = '../models/logistic_weights.npy'

# 1. Load the newly created file
with open(model_path, 'rb') as fd:
    W_loaded = np.load(fd)

# 2. Quick check with the Test set
# (Although training was performed with the entire dataset, the Test set is used to verify the matrix works)
haty_test = LogisticRegressionClassification(X_test, W_loaded)
accuracy = np.sum(haty_test == y_test) / y_test.size

print(f"Model loaded and functional.")
print(f"Final classifier evaluation error: {1.0 - accuracy:.1%}")

Verifying model loading...
Model loaded and functional.
Final classifier evaluation error: 0.0%


# Classify Custom Doodles (Logistic Regression)

In this final section, the model is tested in a real-world scenario.

Unlike the original course script (which depends on external files for Google Colab), a **complete interface using Gradio** has been implemented here.

The system:
1.  Loads the optimal weight matrix ($W$) trained with Logistic Regression.
2.  Processes the drawing by applying a thickening filter to prevent the loss of detail during reduction to 16x16 resolution.
3.  Utilizes the **Softmax** function to calculate the probability of each class and select the winning class.

In [ ]:
import gradio as gr
import numpy as np
from PIL import Image, ImageOps, ImageFilter
import os

# --- 1. CONFIGURATION ---
MODEL_PATH = '../models/logistic_weights.npy'
DOODLE_MAP = {
    0: "Smiley Face", 1: "Heart", 2: "Star", 3: "Lightning",
    4: "Moon",        5: "Cloud",    6: "Tree",   7: "House",
    8: "Sad Face",    9: "Sun"
}

# --- 2. MODEL LOADING ---
if os.path.exists(MODEL_PATH):
    with open(MODEL_PATH, 'rb') as fd:
        W_loaded = np.load(fd)
    print(f"Logistic Regression Model loaded.")
else:
    raise FileNotFoundError("The .npy file is not found. Execute the training above!")

# --- 3. PREDICTION LOGIC (WITH ROBUST PREPROCESSING) ---
def predict_logistic(image_input):
    if image_input is None: return "Draw something first."

    # A. Gradio input management
    image = image_input.get("composite") if isinstance(image_input, dict) else image_input

    # B. Preprocessing ('Thick Marker' Trick)
    img = image.convert("L")
    img = ImageOps.invert(img)
    # Stroke is thickened so it does not disappear upon reduction
    img = img.filter(ImageFilter.MaxFilter(9))
    img = img.resize((16, 16), resample=Image.BILINEAR)

    # C. Prepare data for the model
    # Convert to array and flatten -> Shape (1, 256)
    x = np.array(img)
    x_flat = x.flatten().reshape(1, -1)

    # D. Inference (Logistic Regression Mathematics)
    # The function 'LogisticRegressionClassification' defined previously is used
    # If an error stating it does not exist occurs, uncomment the lines below to perform it manually:
    """
    N, D = x_flat.shape
    X_aug = np.concatenate((np.ones((N,1)), x_flat), axis=1) # Add bias
    logits = X_aug @ W_loaded
    logits -= logits.max(axis=1, keepdims=True) # Stability
    probs = np.exp(logits) / np.sum(np.exp(logits), axis=1, keepdims=True)
    pred_idx = np.argmax(probs, axis=1)[0]
    confidence = probs[0][pred_idx]
    """
    # Direct call to the function (ensure the functions cell above has been executed)
    pred_idx = LogisticRegressionClassification(x_flat, W_loaded)[0]

    return f"Prediction: {DOODLE_MAP.get(pred_idx, 'Unknown')}"

# --- 4. LAUNCH INTERFACE ---
interface = gr.Interface(
    fn=predict_logistic,
    inputs=gr.Sketchpad(label="Draw here (Slow and clear stroke)", type="pil", brush=gr.Brush(default_size=20)),
    outputs="text",
    title="Doodle Judge (Logistic Regression)",
    description="Draw and let the probability model decide what it is."
)

interface.launch(debug=True)

Logistic Regression Model loaded.
* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
